# Friends Demo — Phase 2: Generation, Conflict Resolution, Layer Scoping

Continuing from the feature showcase notebook -- same `client`, same Maya/Jordan/Sam data.
Three things get built here:

1. **Generation** -- turn retrieved memories into an actual answer, not just a printed list
2. **Conflict resolution** -- Maya switches hobbies, and we watch (or don't watch) mem0 reconcile it
3. **Layer scoping** -- `user_id` / `run_id` / `agent_id` filters, using this friend group


## 0. Reconnect and confirm starting data

In [ ]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

for user_id in ["maya", "jordan", "sam"]:
    print(f"--- {user_id} ---")
    memories = client.get_all(filters={"user_id": user_id})
    for item in memories.get("results", []):
        print(" -", item["memory"])
    print()

## 1. Generation -- answer a question using memory

mem0 retrieves facts, it doesn't answer questions. We still need our own LLM call for that,
using the lab's config from earlier notebooks.


In [ ]:
from lab_llm_config import complete

def answer_with_memory(question, user_id):
    results = client.search(query=question, filters={"user_id": user_id})
    memories = [r["memory"] for r in results.get("results", [])]

    memory_block = "\n".join(f"- {m}" for m in memories)
    prompt = (
        "You're chatting with a friend. Use these known facts about them if relevant, "
        "and answer naturally without mentioning 'stored memories':\n\n"
        f"{memory_block}\n\nQuestion: {question}"
    )
    answer = complete(prompt)
    return answer, memories

In [ ]:
question = "What should I get Maya for her birthday?"
answer, used_memories = answer_with_memory(question, user_id="maya")

print("Memories used:")
for m in used_memories:
    print(" -", m)

print("\nAnswer:")
print(answer)

### Store this exchange, same as any other turn

In [ ]:
client.add(
    [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ],
    user_id="maya",
)
print("Stored.")

## 2. Conflict resolution -- Maya changes hobbies

She stops doing pottery and takes up painting instead. Snapshot before and after, so the
change (or lack of one) is visible, not assumed.


In [ ]:
before = client.get_all(filters={"user_id": "maya"})
before_facts = [item["memory"] for item in before.get("results", [])]

In [ ]:
add_result = client.add(
    "Actually, I quit pottery a few weeks ago -- I've switched to painting instead, "
    "I go to a studio on Thursdays now.",
    user_id="maya",
)
print(add_result)

In [ ]:
after = client.get_all(filters={"user_id": "maya"})
after_facts = [item["memory"] for item in after.get("results", [])]

print("BEFORE:")
for f in before_facts:
    print(" -", f)

print("\nAFTER:")
for f in after_facts:
    print(" -", f)

### Confirm at the retrieval level

Does search correctly favor painting over pottery when asked directly?


In [ ]:
results = client.search(query="what hobby is Maya doing these days?", filters={"user_id": "maya"})
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

Look at what's actually there before drawing a conclusion. If the pottery fact is still
sitting in storage unchanged, that's the same finding as the aircraft example -- storage adds,
it doesn't reliably reconcile. Whatever these three cells show is the real answer.


## 3. Layer scoping with `user_id` / `run_id` / `agent_id`

| Layer | mem0 filter |
|---|---|
| User (durable) | `user_id` only |
| Session (ephemeral) | `user_id` **and** `run_id` |
| Agent (behavior rule) | `agent_id` |

### 3a. A session-scoped fact

Something that only matters to today's conversation with Jordan, not their permanent record.


In [ ]:
client.add(
    "Jordan is stressed today about planning the jazz-themed dinner party.",
    user_id="jordan",
    run_id="chat_today",
)
print("Session-scoped fact stored under run_id='chat_today'.")

### 3b. Does a plain `user_id` search pick it up?

In [ ]:
results_user_only = client.search(
    query="how is Jordan feeling about the dinner party",
    filters={"user_id": "jordan"},
)
print("user_id only:")
for r in results_user_only.get("results", []):
    print(" -", r["memory"])

In [ ]:
results_with_run = client.search(
    query="how is Jordan feeling about the dinner party",
    filters={"AND": [{"user_id": "jordan"}, {"run_id": "chat_today"}]},
)
print("user_id + run_id:")
for r in results_with_run.get("results", []):
    print(" -", r["memory"])

### 3c. An agent-layer fact

Not about any specific friend -- a behavior rule for the assistant itself. Added standalone
since it doesn't naturally arise from the conversation.

**Heads up:** combining `user_id` + `agent_id` filters together is a known flaky case in mem0
-- run the cells below and see what you actually get.


In [ ]:
client.add(
    "Always suggest specific, personal gift ideas -- never generic suggestions like 'a gift card'.",
    agent_id="friend_assistant",
)
print("Agent-layer fact stored.")

In [ ]:
by_agent_only = client.get_all(filters={"agent_id": "friend_assistant"})
print("agent_id only:")
for item in by_agent_only.get("results", []):
    print(" -", item["memory"])

In [ ]:
by_user_and_agent = client.get_all(
    filters={"AND": [{"user_id": "maya"}, {"agent_id": "friend_assistant"}]}
)
print("user_id + agent_id combined:")
for item in by_user_and_agent.get("results", []):
    print(" -", item["memory"])
print("\n(Empty here matches the known mem0 filter-combination issue -- not a bug in your code.)")

## Wrap-up

1. **Generation** turns retrieval into an actual usable answer.
2. Maya's pottery-to-painting switch is the same test as the aircraft engineer's fleet change
   -- whatever it showed here should match what you found there, since it's testing the same
   mechanism against a different story.
3. `run_id` and `agent_id` scoping behave the way this notebook's cells actually showed --
   not the way the docs describe in the abstract.
